# IMDb Movie Rating Prediction

**Portfolio reconstruction of December 2025 coursework.** The originally submitted notebook was unavailable. Historical R² = 0.443 is *not* generated by this notebook; it must not be confused with its newly computed scores.

## 1. Import libraries and load the CSV
Download the IMDb Top 1000 dataset as `data/imdb_top_1000.csv` first, and run this notebook from the repository root.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.train import FEATURES, TARGET, load_and_clean, fit_explanatory_ols, fit_holdout_model

data_path = Path('data/imdb_top_1000.csv')
assert data_path.exists(), 'Download the IMDb Top 1000 CSV into data/imdb_top_1000.csv first.'
df = load_and_clean(data_path)
display(df[[TARGET, *FEATURES]].head())
display(df[[TARGET, *FEATURES]].describe().T)
display(df[[TARGET, *FEATURES]].isna().sum().rename('missing_values'))

## 2. Exploratory data analysis
Explore distributions and numeric correlations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(df[TARGET].astype(float).to_numpy(), bins=25, edgecolor='white', alpha=0.82)
axes[0].set_title('IMDb rating distribution')
sns.heatmap(df[[TARGET, *FEATURES]].corr(numeric_only=True), cmap='coolwarm', center=0, annot=True, fmt='.2f', ax=axes[1])
axes[1].set_title('Numeric correlations')
plt.tight_layout()
plt.show()

## 3. Explanatory full-sample OLS
The following model uses median-imputed predictors and produces an **in-sample** model summary. It is not an out-of-sample prediction score.

In [ ]:
ols = fit_explanatory_ols(df)
print(ols.summary())

## 4. Holdout predictive evaluation
Train/test imputation is fit only on training data. Report test R², MAE, and RMSE separately from full-sample fit.

In [ ]:
model, test_metrics, actual, predicted = fit_holdout_model(df, test_size=0.2, random_state=42)
display(pd.DataFrame([test_metrics]))
sns.scatterplot(x=actual, y=predicted, alpha=0.65)
low = min(min(actual), min(predicted))
high = max(max(actual), max(predicted))
plt.plot([low, high], [low, high], 'k--')
plt.xlabel('Actual IMDb rating')
plt.ylabel('Predicted IMDb rating')
plt.title('Held-out: actual vs. predicted')
plt.show()

## 5. Interpretation and caveats

- Historical reported in-sample R² = **0.443** was not independently reproduced here.
- Newly calculated holdout scores measure generalization on held-out movies.
- Associations are not necessarily causal.
- IMDb Top 1000 is a selected high-rated sample; votes, revenue, and Metascore may only be available after release.
- Missing-value rules and source data version may differ from the coursework.